# Курсовая — чистый пайплайн (рефакторинг)

**Оптимизация кредитных решений на основе машинного обучения и экономической функции прибыли.**

Версия после рефакторинга структуры.

### Содержание
- **Блок 0.** Импорты и конфигурация.
- **Блок 1.** Подготовка данных (загрузка, survival-разметка, канонические срезы, признаки). Без моделей.
- **Блок 1.5.** Диагностика цензурирования (§3.1): доля потерь тела по зрелости.
- **Блок 2.** Наивная постановка — наивная строка Табл. 3.2 (GBM/LR на loan-level).
- **Блок 3.** Hazard-PD@36 — hazard-строка Табл. 3.2 (GBM/LR, дискретный хазард).
- **Блок 4.** Калибровка PD@36 (изотоническая; Brier/ECE/KS; сравнение калибраторов).

Блок 5 (денежная модель / H2) и проверки §3.4.1, §3.5, §3.6, ablation, кластерный вывод —
следующий шаг: они навешиваются на канонические объекты этого ядра (`pd36_gbm_cal`, `surv_test`).

**Запуск:** свежее ядро, строго сверху вниз. Требуется `../data/accepted_2007_to_2018Q4.csv`.
Зерно фиксировано (`SEED=42`).

## Блок 0. Импорты и конфигурация

In [1]:
import warnings; warnings.filterwarnings('ignore')   # доброкачественные предупреждения sklearn/LightGBM
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, brier_score_loss
np.seterr(all='ignore')
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 40)

SEED = 42                                  # единое зерно всех источников случайности
H = 36                                     # горизонт оценки вероятности дефолта, мес.
SNAPSHOT = pd.Timestamp('2018-12-31')      # дата среза дампа accepted_2007_to_2018Q4
DATA = '../data/accepted_2007_to_2018Q4.csv'

## Блок 1. Подготовка данных
Только загрузка, разметка и разбиение — без моделей. Создаются: `surv` (survival-кадр со всеми
пригодными кредитами, незавершённые — как цензурированные), `df_completed` (наивная loan-level
выборка завершённых) и канонические срезы по годам.

In [2]:
# Единая загрузка. surv = все пригодные кредиты (живые + завершённые):
#   event=1  — наблюдён дефолт (Charged Off / Default);
#   mob      — месяцев на книге к дате среза;
#   duration — прожитые месяцы (прокси total_pymnt/installment), ограниченные сроком кредита;
# незавершённые входят как правосторонне цензурированные (нужно для hazard-PD и денежной метрики).
NEEDED = ['id','loan_status','issue_d','loan_amnt','int_rate','term','installment','grade','sub_grade',
 'emp_length','home_ownership','annual_inc','verification_status','purpose','addr_state','dti',
 'delinq_2yrs','earliest_cr_line','fico_range_low','fico_range_high','inq_last_6mths','open_acc',
 'pub_rec','revol_bal','revol_util','total_acc','application_type','mort_acc','pub_rec_bankruptcies',
 'total_pymnt','recoveries','total_rec_int','total_rec_prncp']
surv = pd.read_csv(DATA, usecols=lambda c: c in NEEDED, low_memory=False)
surv = surv[surv['loan_status'].notna()].copy()
surv['int_rate']    = pd.to_numeric(surv['int_rate'].astype(str).str.rstrip('%').str.strip(), errors='coerce')
surv['term_months'] = pd.to_numeric(surv['term'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
surv['issue_date']  = pd.to_datetime(surv['issue_d'], format='%b-%Y', errors='coerce')
surv = surv[surv['issue_date'].notna() & surv['term_months'].notna() & surv['installment'].gt(0)].copy()
surv['issue_year']  = surv['issue_date'].dt.year

DEFAULT_STATUS = ['Charged Off','Default','Does not meet the credit policy. Status:Charged Off']
PAID_STATUS    = ['Fully Paid','Does not meet the credit policy. Status:Fully Paid']
surv['event']    = surv['loan_status'].isin(DEFAULT_STATUS).astype(int)
surv['mob']      = ((SNAPSHOT.year-surv['issue_date'].dt.year)*12
                    + (SNAPSHOT.month-surv['issue_date'].dt.month)).clip(lower=0)
surv['mpaid']    = (surv['total_pymnt'] / surv['installment']).round()
surv['duration'] = np.where(surv['loan_status'].isin(DEFAULT_STATUS+PAID_STATUS), surv['mpaid'], surv['mob'])
surv['duration'] = surv['duration'].clip(lower=1).combine(surv['term_months'], min)   # не больше срока
print(f'surv: {len(surv):,} кредитов | наблюдённых дефолтов {surv["event"].mean()*100:.1f}%')

surv: 2,260,668 кредитов | наблюдённых дефолтов 11.9%


In [3]:
# Наивная (loan-level) выборка для наивной строки Табл. 3.2: ТОЛЬКО завершённые кредиты,
# без контроля зрелости; цель — пожизненный дефолт. Незрелые кредиты здесь НЕ отбрасываются,
# что и даёт завышенный AUC (часть сигнала — артефакт цензурирования, см. §3.1/§3.2 работы).
NAIVE_COMPLETED = ['Fully Paid','Charged Off','Default']
df_completed = surv[surv['loan_status'].isin(NAIVE_COMPLETED)].copy()
df_completed['default'] = df_completed['loan_status'].isin(['Charged Off','Default']).astype(int)
print(f'df_completed (наивная): {len(df_completed):,} | дефолтов {df_completed["default"].mean()*100:.1f}%')

df_completed (наивная): 1,345,350 | дефолтов 20.0%


In [4]:
# Признаки уровня кредита (известны на момент заявки; post-application поля — total_pymnt,
# recoveries, total_rec_* — НЕ входят в признаки, чтобы исключить утечку, и используются только
# в §3.1 и в денежной метрике Блока 5).
NUM_FEATURES = ['loan_amnt','int_rate','term_months','annual_inc','dti','delinq_2yrs','fico_range_low',
    'fico_range_high','inq_last_6mths','open_acc','pub_rec','revol_bal','revol_util','total_acc',
    'mort_acc','pub_rec_bankruptcies']
CAT_FEATURES = ['grade','sub_grade','emp_length','home_ownership','verification_status','purpose','application_type']

def build_loan_matrix(df, base_cols):
    """Матрица признаков уровня кредита: числовые + one-hot категориальные.
    base_cols=None — задать опорный набор колонок по обучающей выборке; иначе выровнять по нему
    (reindex с нулями), чтобы train/valid/test совпадали по столбцам."""
    X = pd.concat([df[NUM_FEATURES].reset_index(drop=True),
                   pd.get_dummies(df[CAT_FEATURES].astype(str).reset_index(drop=True), drop_first=True)], axis=1)
    return X if base_cols is None else X.reindex(columns=base_cols, fill_value=0)

def y_default_within_H(df):
    """Целевая PD@36: дефолт наблюдён И произошёл в пределах горизонта H месяцев."""
    return ((df['event'].values == 1) & (df['duration'].values <= H)).astype(int)

# Канонические срезы (определяются ОДИН раз и далее только читаются — никаких переопределений te/tr):
surv_train  = surv[surv['issue_year'].isin([2012, 2013])]    # обучение hazard/LR
surv_valid  = surv[surv['issue_year'] == 2014]               # калибровка
surv_test   = surv[surv['issue_year'] == 2015]               # тест
naive_train = df_completed[df_completed['issue_year'].isin([2012, 2013])]
naive_test  = df_completed[df_completed['issue_year'] == 2015]
print(f'surv:  train={len(surv_train):,}  valid={len(surv_valid):,}  test={len(surv_test):,}')
print(f'naive: train={len(naive_train):,}  test={len(naive_test):,}')

surv:  train=188,181  valid=235,629  test=421,095
naive: train=188,171  test=375,546


## Блок 1.5. Диагностика цензурирования (§3.1, Рис. 3.1 / Табл. 3.1)
Реализованная доля потерь тела по дефолтным кредитам в разрезе «год выдачи × зрелость».

In [5]:
diag = surv.copy()
diag['matured']   = diag['mob'] >= diag['term_months']
diag['loss_body'] = ((diag['loan_amnt'] - diag['total_rec_prncp'] - diag['recoveries'].fillna(0))
                     / diag['loan_amnt']).clip(0, 1)
co = diag['event'] == 1                                  # только дефолтные кредиты
print('Доля потерь тела по году выдачи × зрелость (False=незрелые, True=зрелые):')
print(diag[co].groupby(['issue_year','matured'])['loss_body'].mean()
      .unstack('matured').round(3).loc[2013:2018].to_string())
print('\nСредние recoveries на дефолтный кредит по году выдачи:')
print(diag[co].groupby('issue_year')['recoveries'].mean().round(0).loc[2013:2018].to_string())

Доля потерь тела по году выдачи × зрелость (False=незрелые, True=зрелые):
matured     False  True 
issue_year              
2013          NaN  0.545
2014        0.620  0.495
2015        0.678  0.508
2016        0.633    NaN
2017        0.751    NaN
2018        0.890    NaN

Средние recoveries на дефолтный кредит по году выдачи:
issue_year
2013    1323.0
2014    1355.0
2015    1301.0
2016    1210.0
2017    1081.0
2018     555.0


In [6]:
# Зависимости: Блок 1 (surv, NAIVE_COMPLETED)
comp = surv[surv['loan_status'].isin(NAIVE_COMPLETED)].copy()
comp['matured'] = comp['mob'] >= comp['term_months']
share = (1 - comp.groupby('issue_year')['matured'].mean()).loc[2014:2018].round(2)
print("Доля незрелых среди завершённых по году выдачи:"); print(share.to_string())

Доля незрелых среди завершённых по году выдачи:
issue_year
2014    0.27
2015    0.25
2016    1.00
2017    1.00
2018    1.00


## Блок 2. Наивная постановка — наивная строка Табл. 3.2
Loan-level классификатор пожизненного дефолта на завершённых кредитах (без контроля зрелости).
Имена с суффиксом `_naive`, чтобы их нельзя было перепутать с hazard-объектами.

In [7]:
# Матрицы наивной постановки (опорные колонки — по обучающей выборке).
Xtr_naive  = build_loan_matrix(naive_train, None)
naive_cols = list(Xtr_naive.columns)
Xte_naive  = build_loan_matrix(naive_test, naive_cols)
ytr_naive  = naive_train['default'].values
yte_naive  = naive_test['default'].values
assert Xtr_naive.shape[1] == Xte_naive.shape[1] == len(naive_cols)
assert len(Xtr_naive) == len(ytr_naive) and len(Xte_naive) == len(yte_naive)
print(f'naive: Xtr={Xtr_naive.shape}, Xte={Xte_naive.shape}')

# GBM наивной постановки
gbm_naive = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=100, random_state=SEED, verbose=-1).fit(Xtr_naive, ytr_naive)
pred_gbm_naive = gbm_naive.predict_proba(Xte_naive)[:, 1]

# LR наивной постановки (импутация медианой + стандартизация; nan_to_num защищает от вырожденных
# one-hot столбцов с нулевой дисперсией, дающих inf после стандартизации)
imp_naive = SimpleImputer(strategy='median').fit(Xtr_naive.values.astype(float))
sc_naive  = StandardScaler().fit(imp_naive.transform(Xtr_naive.values.astype(float)))
std_naive = lambda X: np.nan_to_num(sc_naive.transform(imp_naive.transform(X.values.astype(float))),
                                    nan=0., posinf=0., neginf=0.)
lr_naive = LogisticRegression(solver='lbfgs', max_iter=2000, C=1.0, random_state=SEED)
lr_naive.fit(std_naive(Xtr_naive), ytr_naive)
pred_lr_naive = lr_naive.predict_proba(std_naive(Xte_naive))[:, 1]

auc_gbm_naive = roc_auc_score(yte_naive, pred_gbm_naive)
auc_lr_naive  = roc_auc_score(yte_naive, pred_lr_naive)

# Бутстрэп-ДИ для ΔAUC (GBM − LR) на уровне кредитов
rng = np.random.default_rng(SEED); Nn = len(yte_naive); dboot = []
for _ in range(2000):
    ix = rng.integers(0, Nn, Nn)
    if yte_naive[ix].min() == yte_naive[ix].max(): continue
    dboot.append(roc_auc_score(yte_naive[ix], pred_gbm_naive[ix]) - roc_auc_score(yte_naive[ix], pred_lr_naive[ix]))
lo, hi = np.percentile(dboot, [2.5, 97.5])
print(f'НАИВНАЯ (loan-level, тест 2015, n={Nn:,}): GBM={auc_gbm_naive:.4f}  LR={auc_lr_naive:.4f}')
print(f'  ΔAUC(GBM-LR) = {auc_gbm_naive-auc_lr_naive:+.4f}  95%CI[{lo:+.4f}; {hi:+.4f}]')

naive: Xtr=(188171, 85), Xte=(375546, 85)
НАИВНАЯ (loan-level, тест 2015, n=375,546): GBM=0.7181  LR=0.7262
  ΔAUC(GBM-LR) = -0.0081  95%CI[-0.0091; -0.0072]


## Блок 3. Hazard-PD@36 — hazard-строка Табл. 3.2
Дискретная hazard-модель: кредиты разворачиваются в «кредит-месяц», PD@36 собирается как
1 − Π(1 − hazard). Имена с суффиксом `_hazard` / `pd36_*`. Калибровка вынесена в Блок 4.

In [8]:
def expand_person_period(df, base_cols, q=0.15, seed=SEED):
    """Разворот кредитов в «кредит-месяц» до min(дефолт, H). Сохраняем ВСЕ месяцы-события и
    подвыборку не-событий с долей q; веса 1/q возвращают несмещённость хазарда (case-control).
    Последний столбец матрицы — mob (месяц на книге), задающий форму базового хазарда."""
    df = df.reset_index(drop=True); dur = df['duration'].values
    ev_H = ((df['event'].values == 1) & (dur <= H)).astype(int)   # событие только в пределах горизонта
    T = np.clip(np.minimum(dur, H).astype(int), 1, H)
    idx = np.repeat(np.arange(len(df)), T)
    mob = np.ones(int(T.sum()), int); mob[np.cumsum(T)[:-1]] -= T[:-1]; mob = np.cumsum(mob)
    y = ((mob == np.repeat(T, T)) & (np.repeat(ev_H, T) == 1)).astype(np.int8)
    rng_e = np.random.default_rng(seed)
    keep = (y == 1) | (rng_e.random(len(y)) < q)
    w = np.where(y[keep] == 1, 1.0, 1.0/q).astype(np.float32)
    Xl = build_loan_matrix(df, base_cols).astype(np.float32).values
    Xpp = np.column_stack([Xl[idx[keep]], mob[keep].astype(np.float32)])
    return Xpp, y[keep], w

hazard_cols = list(build_loan_matrix(surv_train, None).columns)        # опорные колонки признаков
Xtr_hazard, ytr_hazard, wtr_hazard = expand_person_period(surv_train, hazard_cols)
Xva_hazard, yva_hazard, wva_hazard = expand_person_period(surv_valid, hazard_cols)
assert Xtr_hazard.shape[1] == Xva_hazard.shape[1] == len(hazard_cols) + 1   # +1 = столбец mob
print(f'person-period: train={Xtr_hazard.shape}, valid={Xva_hazard.shape}, доля событий={ytr_hazard.mean()*100:.2f}%')

# GBM-hazard (ранняя остановка по валидации 2014)
gbm_hazard = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
    min_child_samples=200, random_state=SEED, verbose=-1)
gbm_hazard.fit(Xtr_hazard, ytr_hazard, sample_weight=wtr_hazard,
    eval_set=[(Xva_hazard, yva_hazard)], eval_sample_weight=[wva_hazard],
    callbacks=[lgb.early_stopping(50, verbose=False)])

# LR-hazard (импутация + стандартизация; решатель saga — как в работе)
imp_hazard = SimpleImputer(strategy='median').fit(Xtr_hazard)
sc_hazard  = StandardScaler().fit(imp_hazard.transform(Xtr_hazard))
lr_hazard  = LogisticRegression(solver='saga', max_iter=200, n_jobs=-1, random_state=SEED)
lr_hazard.fit(sc_hazard.transform(imp_hazard.transform(Xtr_hazard)), ytr_hazard, sample_weight=wtr_hazard)

def survival_pd36(month_hazard, df):
    """PD@36 = 1 - произведение (1 - hazard_t) по t=1..H. month_hazard(X) — P(дефолт в месяце t)."""
    Xl = build_loan_matrix(df, hazard_cols).astype(np.float32).values; nrow = len(df); surv_p = np.ones(nrow)
    for t in range(1, H+1):
        surv_p *= (1 - month_hazard(np.column_stack([Xl, np.full(nrow, t, np.float32)])))
    return 1 - surv_p

pd36_gbm = survival_pd36(lambda X: gbm_hazard.predict_proba(X)[:, 1], surv_test)
pd36_lr  = survival_pd36(lambda X: lr_hazard.predict_proba(sc_hazard.transform(imp_hazard.transform(X)))[:, 1], surv_test)
y36_test = y_default_within_H(surv_test)
assert len(pd36_gbm) == len(pd36_lr) == len(y36_test) == len(surv_test)

auc_gbm_hazard = roc_auc_score(y36_test, pd36_gbm)
auc_lr_hazard  = roc_auc_score(y36_test, pd36_lr)
print(f'\nHAZARD PD@36 (тест 2015, n={len(y36_test):,}): средняя PD={pd36_gbm.mean()*100:.1f}%  '
      f'факт дефолт@36={y36_test.mean()*100:.1f}%')
print(f'  GBM={auc_gbm_hazard:.4f}  LR={auc_lr_hazard:.4f}')
for trm in (36, 60):
    msk = surv_test['term_months'].values == trm
    if msk.sum() > 500:
        print(f'  AUC@36 срок {trm} мес.: {roc_auc_score(y36_test[msk], pd36_gbm[msk]):.4f}  (n={int(msk.sum()):,})')

person-period: train=(950017, 86), valid=(1188948, 86), доля событий=2.82%

HAZARD PD@36 (тест 2015, n=421,095): средняя PD=13.3%  факт дефолт@36=17.3%
  GBM=0.6900  LR=0.6875
  AUC@36 срок 36 мес.: 0.6839  (n=283,173)
  AUC@36 срок 60 мес.: 0.6728  (n=137,922)


In [9]:
# Бутстрэп-ДИ для ΔAUC (GBM - LR) и тест эквивалентности TOST (порог ±0.005 AUC)
rng = np.random.default_rng(SEED); Nh = len(y36_test); dboot_h = []
for _ in range(2000):
    ix = rng.integers(0, Nh, Nh)
    dboot_h.append(roc_auc_score(y36_test[ix], pd36_gbm[ix]) - roc_auc_score(y36_test[ix], pd36_lr[ix]))
lo95, hi95 = np.percentile(dboot_h, [2.5, 97.5])
lo90, hi90 = np.percentile(dboot_h, [5, 95])
delta = auc_gbm_hazard - auc_lr_hazard
print(f'ΔAUC(GBM-LR) hazard = {delta:+.4f}  95%CI[{lo95:+.4f}; {hi95:+.4f}]')
equiv = (lo90 > -0.005) and (hi90 < 0.005)
print(f'TOST ±0.005: 90% ДИ [{lo90:+.4f}; {hi90:+.4f}] -> '
      f'{"практическая эквивалентность" if equiv else "эквивалентность не доказана"}')

ΔAUC(GBM-LR) hazard = +0.0024  95%CI[+0.0016; +0.0033]
TOST ±0.005: 90% ДИ [+0.0017; +0.0031] -> практическая эквивалентность


### Важность признаков GBM-hazard (диагностика)

In [10]:
# Топ-15 признаков по встроенной метрике важности LightGBM (gain). Диагностика для §3.2: mob и финансовые признаки доминируют.
import pandas as pd
fi = pd.Series(gbm_hazard.feature_importances_, index=hazard_cols+['mob']).sort_values(ascending=False)
print(fi.head(15).to_string())

mob                       1047
annual_inc                 973
revol_bal                  860
loan_amnt                  814
dti                        739
int_rate                   701
revol_util                 652
total_acc                  452
fico_range_low             408
open_acc                   408
mort_acc                   347
inq_last_6mths             284
purpose_small_business     129
term_months                102
home_ownership_RENT         84


## Блок 4. Калибровка PD@36
Изотоническая регрессия учится на валидации 2014 и применяется к тесту 2015. Будучи монотонной,
она не меняет ранжирование (AUC/KS), но снижает ошибку калибровки (Brier/ECE).
Канонический выход — `pd36_gbm_cal` (вход для денежной модели Блока 5).

In [11]:
y36_valid      = y_default_within_H(surv_valid)
pd36_gbm_valid = survival_pd36(lambda X: gbm_hazard.predict_proba(X)[:, 1], surv_valid)
iso_calibrator = IsotonicRegression(out_of_bounds='clip').fit(pd36_gbm_valid, y36_valid)
pd36_gbm_cal   = iso_calibrator.transform(pd36_gbm)
assert len(pd36_gbm_cal) == len(pd36_gbm)

def ece(y, p, bins=10):
    """Expected Calibration Error: средняя |факт − прогноз| по 10 равным бинам вероятности."""
    edges = np.linspace(0, 1, bins+1); e = 0.0; nn = len(y)
    for lo, hi in zip(edges[:-1], edges[1:]):
        msk = (p >= lo) & ((p < hi) if hi < 1 else (p <= hi))
        if msk.sum(): e += msk.sum()/nn * abs(y[msk].mean() - p[msk].mean())
    return e

def ks_stat(y, p):
    """Колмогоров–Смирнов между распределениями скоров дефолтных и недефолтных."""
    o = np.argsort(p); ys = y[o]
    cb = np.cumsum(ys)/max(ys.sum(), 1); cg = np.cumsum(1-ys)/max((1-ys).sum(), 1)
    return float(np.max(np.abs(cb - cg)))

print(f"{'метрика':>7} | {'до калибр.':>10} | {'после изот.':>11}")
print(f"{'Brier':>7} | {brier_score_loss(y36_test, pd36_gbm):>10.4f} | {brier_score_loss(y36_test, pd36_gbm_cal):>11.4f}")
print(f"{'ECE':>7} | {ece(y36_test, pd36_gbm):>10.4f} | {ece(y36_test, pd36_gbm_cal):>11.4f}")
print(f"{'KS':>7} | {ks_stat(y36_test, pd36_gbm):>10.4f} | {ks_stat(y36_test, pd36_gbm_cal):>11.4f}")
print(f"{'AUC':>7} | {roc_auc_score(y36_test, pd36_gbm):>10.4f} | {roc_auc_score(y36_test, pd36_gbm_cal):>11.4f}")
print('Монотонная калибровка не меняет AUC/KS, корректирует Brier/ECE.')

метрика | до калибр. | после изот.
  Brier |     0.1356 |      0.1345
    ECE |     0.0404 |      0.0268
     KS |     0.2757 |      0.2755
    AUC |     0.6900 |      0.6898
Монотонная калибровка не меняет AUC/KS, корректирует Brier/ECE.


In [12]:
# Сравнение калибраторов (изотоническая / Платт / Beta) на тесте 2015
def beta_feats(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps); return np.column_stack([np.log(p), -np.log(1-p)])
platt_cal = LogisticRegression(solver='lbfgs', max_iter=5000, random_state=SEED).fit(pd36_gbm_valid.reshape(-1,1), y36_valid)
beta_cal  = LogisticRegression(solver='lbfgs', max_iter=5000, random_state=SEED).fit(beta_feats(pd36_gbm_valid), y36_valid)
variants = [('без калибровки', pd36_gbm),
            ('изотоническая',  pd36_gbm_cal),
            ('Платт',          platt_cal.predict_proba(pd36_gbm.reshape(-1,1))[:,1]),
            ('Beta',           beta_cal.predict_proba(beta_feats(pd36_gbm))[:,1])]
print(f"{'калибратор':>16} | {'Brier':>7} | {'ECE':>7} | {'AUC':>7} | {'ср.PD,%':>8}")
for nm, p in variants:
    print(f'{nm:>16} | {brier_score_loss(y36_test, p):>7.4f} | {ece(y36_test, p):>7.4f} | '
          f'{roc_auc_score(y36_test, p):>7.4f} | {p.mean()*100:>8.2f}')

      калибратор |   Brier |     ECE |     AUC |  ср.PD,%
  без калибровки |  0.1356 |  0.0404 |  0.6900 |    13.25
   изотоническая |  0.1345 |  0.0268 |  0.6898 |    14.60
           Платт |  0.1353 |  0.0342 |  0.6900 |    14.91
            Beta |  0.1344 |  0.0268 |  0.6900 |    14.59


In [13]:
# Зависимости: Блок 4 (pd36_gbm, pd36_gbm_cal, y36_test, ece, brier_score_loss)
rng = np.random.default_rng(SEED); N = len(y36_test); dB, dE = [], []
for _ in range(2000):
    ix = rng.integers(0, N, N); yb = y36_test[ix]
    dB.append(brier_score_loss(yb, pd36_gbm_cal[ix]) - brier_score_loss(yb, pd36_gbm[ix]))
    dE.append(ece(yb, pd36_gbm_cal[ix]) - ece(yb, pd36_gbm[ix]))
dB, dE = np.array(dB), np.array(dE)
b0 = brier_score_loss(y36_test, pd36_gbm_cal) - brier_score_loss(y36_test, pd36_gbm)
e0 = ece(y36_test, pd36_gbm_cal) - ece(y36_test, pd36_gbm)
print(f'ΔBrier = {b0:+.4f}  95%ДИ[{np.percentile(dB,2.5):+.4f}; {np.percentile(dB,97.5):+.4f}]')
print(f'ΔECE   = {e0:+.4f}  95%ДИ[{np.percentile(dE,2.5):+.4f}; {np.percentile(dE,97.5):+.4f}]')

ΔBrier = -0.0011  95%ДИ[-0.0012; -0.0011]
ΔECE   = -0.0136  95%ДИ[-0.0137; -0.0135]


In [14]:
# Зависимости: pd36_gbm_valid, y36_valid, pd36_gbm, y36_test, ece
rng = np.random.default_rng(SEED); Nv = len(y36_valid); B = 200
briers, eces, means = [], [], []
m_run = np.zeros(len(pd36_gbm)); M2 = np.zeros(len(pd36_gbm)); cnt = 0      # Welford для sd на кредит
for _ in range(B):
    ix = rng.integers(0, Nv, Nv)
    iso_b = IsotonicRegression(out_of_bounds='clip').fit(pd36_gbm_valid[ix], y36_valid[ix])
    pred = iso_b.transform(pd36_gbm)
    briers.append(brier_score_loss(y36_test, pred)); eces.append(ece(y36_test, pred)); means.append(pred.mean())
    cnt += 1; d = pred - m_run; m_run += d/cnt; M2 += d*(pred - m_run)
sd_loan = np.sqrt(M2/(cnt-1))
print(f'Brier  [{min(briers):.4f}; {max(briers):.4f}]')
print(f'ECE    [{min(eces):.4f}; {max(eces):.4f}]')
print(f'ср.PD  [{min(means)*100:.1f}%; {max(means)*100:.1f}%]')
print(f'ср. sd калибр. PD на кредит = {sd_loan.mean()*100:.2f} п.п.')

Brier  [0.1344; 0.1346]
ECE    [0.0253; 0.0284]
ср.PD  [14.4%; 14.8%]
ср. sd калибр. PD на кредит = 0.43 п.п.


---
## Что дальше: Блок 5 и проверки
Это ядро (Блоки 1–4) даёт наивную и hazard-строки Табл. 3.2, калибровку и §3.1. Следующий шаг —
навесить на канонические объекты (`surv`, `surv_test`, `pd36_gbm_cal`) и **не переопределяя их**:
- **Блок 5 (денежная модель, H2, §3.4):** дозревшая денежная выборка (`money_*`), `cashflow_good/bad`,
  `profit_score`/`risk_score`, бюджетная таблица; вырождение допуска (§3.3).
- **Проверки:** риск-скорректированное сравнение (§3.4.1), репликация 2014 (§3.5), H3 (§3.6),
  ablation источника выигрыша (Табл. 3.6), кластерный вывод, 60-мес сплит, перебор гиперпараметров.

## Блок 5. Денежная модель — вырождение допуска (§3.3) и отбор портфеля (H2, §3.4)
Все денежные расчёты идут на **дозревшей** выборке (завершённые И дожившие до срока кредиты) и на
калиброванной hazard-PD `pd36_gbm_cal` из Блока 4. Уникальные имена `money_*`, `profit_score`,
`risk_score`. Реализованный поток `net = total_pymnt + recoveries − loan_amnt` — недисконтированный
(cash-flow proxy, §2.5.1).

In [15]:
# 5.1 Дозревшая денежная выборка + параметрическая функция прибыли
RES_OK = ['Fully Paid','Charged Off','Default',
          'Does not meet the credit policy. Status:Fully Paid',
          'Does not meet the credit policy. Status:Charged Off']

def money_slice(df, pd_cal):
    """Завершённые И дожившие до своего срока кредиты (контроль зрелости обязателен для денежной
    метрики). Возвращает массивы суммы L, ставки r, срока n, реализованного потока net,
    калиброванной PD и факта дефолта@36."""
    mob = df['mob'].values; term = df['term_months'].values
    mask = (mob >= term) & df['loan_status'].isin(RES_OK).values
    L = df['loan_amnt'].values[mask].astype(float)
    return dict(mask=mask, L=L,
        r   = df['int_rate'].values[mask] / 100,
        n   = df['term_months'].values[mask].astype(float),
        net = df['total_pymnt'].values[mask] + df['recoveries'].fillna(0).values[mask] - L,
        pd_cal = pd_cal[mask],
        yv  = ((df['event'].values[mask] == 1) & (df['duration'].values[mask] <= H)).astype(int))

money = money_slice(surv_test, pd36_gbm_cal)
L, r, n_term, net = money['L'], money['r'], money['n'], money['net']
pd_money, yv_money = money['pd_cal'], money['yv']
assert len(L) == len(r) == len(n_term) == len(net) == len(pd_money) == len(yv_money)
print(f"Дозревшая денежная выборка теста 2015: n={money['mask'].sum():,} | "
      f"60-мес среди них {(n_term==60).mean()*100:.1f}% | дефолт@36 {yv_money.mean()*100:.1f}%")

def pv_comp(L, r, n, lgd=0.50, disc=0.03, dmf=0.15):
    """Ожидаемая приведённая прибыль: возвращает (π_good, π_def).
    π_good — полностью выплаченный кредит; π_def — дефолт на месяце m=dmf·срок с возвратом доли
    (1−lgd) остатка тела. disc — ставка дисконтирования δ (стоимость капитала), dmf — момент дефолта."""
    i = r/12.; dd = disc/12.
    A = np.where(i>0, L*i/(1-(1+i)**(-n)), L/n)                 # аннуитетный платёж
    def ann(x, k):
        x = np.asarray(x, float)*np.ones_like(k); return np.where(x>0, (1-(1+x)**(-k))/x, k)
    pvg = A*ann(dd, n) - L
    m  = np.clip(np.floor(dmf*n), 0, n)
    Bm = np.where(i>0, A*(1-(1+i)**(-(n-m)))/i, L*(n-m)/n)      # остаток тела на момент дефолта
    pvd = A*ann(dd, m) + (1-lgd)*Bm*(1+dd)**(-m) - L
    return pvg, pvd

Дозревшая денежная выборка теста 2015: n=283,026 | 60-мес среди них 0.0% | дефолт@36 14.9%


### 5.2 Вырождение задачи допуска (§3.3)
Доля заявок с E[π_i] ≥ 0 при разных δ

In [16]:
print(f"{'δ':>4} | {'доля E[π]>=0':>12} | {'ср.E[π]/заявку, $':>17}")
for disc in [0.00, 0.03, 0.06, 0.09, 0.12]:
    g, b = pv_comp(L, r, n_term, lgd=0.50, disc=disc)
    epi = (1-pd_money)*g + pd_money*b
    print(f"{disc*100:>3.0f}% | {(epi>=0).mean()*100:>11.1f}% | {epi.mean():>17.0f}")

   δ | доля E[π]>=0 | ср.E[π]/заявку, $
  0% |        99.9% |              1394
  3% |        99.1% |               789
  6% |        69.0% |               220
  9% |        18.5% |              -315
 12% |         1.8% |              -818


### 5.3 H2 — отбор портфеля при ограничении капитала (Табл. 3.4)
Заявки упорядочиваются по критерию и набираются в портфель до исчерпания доли бюджета от суммарного
запрошенного объёма. δ = 3 %.

In [17]:
g, b = pv_comp(L, r, n_term, lgd=0.50, disc=0.03)
profit_score = ((1-pd_money)*g + pd_money*b) / L          # ожидаемая прибыль на доллар (больше = лучше)
risk_score   = pd_money                                    # риск-отбор: меньшая PD первой
rng = np.random.default_rng(SEED); tot = L.sum()
orders = {'profit'  : np.argsort(-profit_score),
          'risk(PD)': np.argsort(risk_score),
          'rate'    : np.argsort(-r),
          'random'  : rng.permutation(len(L))}

def port_flow(order, f):
    """Реализованный поток (млн $) портфеля, набранного по order до доли бюджета f."""
    sel = order[np.cumsum(L[order]) <= f*tot]
    return net[sel].sum()/1e6

print(f"{'бюджет':>7} | {'profit':>8} | {'risk(PD)':>8} | {'rate':>8} | {'random':>8}   (млн $)")
for f in [0.10, 0.30, 0.50, 1.00]:
    print(f"{int(f*100):>6}% | " + ' | '.join(f'{port_flow(o, f):>8.1f}' for o in orders.values()))

 бюджет |   profit | risk(PD) |     rate |   random   (млн $)
    10% |     33.6 |     26.3 |     21.6 |     26.8
    30% |     93.9 |     82.3 |     79.3 |     81.3
    50% |    149.6 |    143.8 |    139.8 |    138.4
   100% |    277.5 |    277.5 |    277.5 |    277.5


In [18]:
# Зависимости: Блок 5 (net, yv_money, L, tot, port_flow)
def flow_by(order, f): s = order[np.cumsum(L[order]) <= f*tot]; return net[s].sum()/1e6
f = 0.30
print(f"hindsight по бинарному исходу (недефолты первыми): {flow_by(np.argsort(yv_money), f):.0f} млн")
print(f"hindsight по реализованному потоку (max net):       {flow_by(np.argsort(-net), f):.0f} млн")
print(f"для сравнения: risk(PD)={port_flow(orders['risk(PD)'], f):.0f}, random={port_flow(orders['random'], f):.0f}")

hindsight по бинарному исходу (недефолты первыми): 155 млн
hindsight по реализованному потоку (max net):       202 млн
для сравнения: risk(PD)=82, random=81


### 5.4 Профиль отобранного портфеля @30 % (вход Табл. 3.5 / §3.4)
Что именно отбирает каждый критерий. Ожидаемо: profit — выше ставка, PD и дефолтность, но и
дисперсия потока; risk — низкие PD/дефолт, но и низкая ставка.

In [19]:
f = 0.30
profile_orders = {'profit (E[π]/$)': np.argsort(-profit_score),
                  'risk (PD)'       : np.argsort(risk_score),
                  'random'          : np.random.default_rng(SEED).permutation(len(L))}
hdr = (f"{'критерий':>16} | {'n':>6} | {'ставка':>7} | {'сумма':>7} | {'ср.PD':>6} | "
       f"{'дефолт@36':>9} | {'net/кред':>9} | {'σ(net)':>8}")
print(hdr); print('-'*len(hdr))
for k, o in profile_orders.items():
    s = o[np.cumsum(L[o]) <= f*tot]
    print(f"{k:>16} | {len(s):>6} | {r[s].mean()*100:>6.1f}% | ${L[s].mean():>6.0f} | "
          f"{pd_money[s].mean()*100:>5.1f}% | {yv_money[s].mean()*100:>8.1f}% | "
          f"${net[s].mean():>+8.0f} | ${net[s].std():>7.0f}")

        критерий |      n |  ставка |   сумма |  ср.PD | дефолт@36 |  net/кред |   σ(net)
-----------------------------------------------------------------------------------------
 profit (E[π]/$) |  83359 |   14.3% | $ 13043 |  14.2% |     18.8% | $   +1126 | $   3929
       risk (PD) |  72155 |    7.3% | $ 15069 |   4.6% |      5.4% | $   +1141 | $   2034
          random |  84925 |   11.3% | $ 12803 |  12.6% |     15.0% | $    +957 | $   3101


### 5.5 Устойчивость выигрыша H2 к LGD и моменту дефолта m (бюджет 30 %)

In [20]:
f = 0.30
base_risk = port_flow(orders['risk(PD)'], f)
print(f'риск-поток (база, PD) @30% = {base_risk:.1f} млн\n--- устойчивость к LGD (m=15%) ---')
for lgd in [0.45, 0.50, 0.55]:
    gg, bb = pv_comp(L, r, n_term, lgd=lgd, disc=0.03, dmf=0.15)
    sp = ((1-pd_money)*gg + pd_money*bb) / L
    pf = port_flow(np.argsort(-sp), f)
    print(f'  LGD={lgd}: profit={pf:.1f}  Δ={pf-base_risk:+.1f} млн')
print('--- устойчивость к моменту дефолта m (LGD=0.50) ---')
for dmf in [0.10, 0.15, 0.25, 0.40]:
    gg, bb = pv_comp(L, r, n_term, lgd=0.50, disc=0.03, dmf=dmf)
    sp = ((1-pd_money)*gg + pd_money*bb) / L
    pf = port_flow(np.argsort(-sp), f)
    print(f'  m={int(dmf*100)}% срока: profit={pf:.1f}  Δ={pf-base_risk:+.1f} млн')

риск-поток (база, PD) @30% = 82.3 млн
--- устойчивость к LGD (m=15%) ---
  LGD=0.45: profit=92.5  Δ=+10.2 млн
  LGD=0.5: profit=93.9  Δ=+11.5 млн
  LGD=0.55: profit=94.6  Δ=+12.3 млн
--- устойчивость к моменту дефолта m (LGD=0.50) ---
  m=10% срока: profit=94.6  Δ=+12.3 млн
  m=15% срока: profit=93.9  Δ=+11.5 млн
  m=25% срока: profit=90.6  Δ=+8.3 млн
  m=40% срока: profit=87.2  Δ=+4.8 млн


### 5.6 Устойчивость ОТБОРА H2 к ставке дисконтирования δ (бюджет 30%).
profit_score считается при δ=3%. Проверяем, что РАНЖИРОВАНИЕ заявок (а значит состав портфеля и выигрыш) почти не зависит от δ: δ в основном масштабирует прибыль, не меняя порядок.

In [21]:
from scipy.stats import spearmanr
f = 0.30
g0, b0 = pv_comp(L, r, n_term, lgd=0.50, disc=0.03)
base_score = ((1 - pd_money) * g0 + pd_money * b0) / L          # базовый критерий (δ=3%)
orr = np.argsort(risk_score)
rf = net[orr[np.cumsum(L[orr]) <= f * tot]].sum() / 1e6         # поток risk(PD)-портфеля
print("  δ  | Spearman с δ=3% | выигрыш над risk(PD), млн $")
for disc in [0.00, 0.03, 0.06, 0.09, 0.12]:
    g, b = pv_comp(L, r, n_term, lgd=0.50, disc=disc)
    sc = ((1 - pd_money) * g + pd_money * b) / L
    rho = spearmanr(sc, base_score).correlation                 # ранговая корреляция с базой
    op = np.argsort(-sc); pf = net[op[np.cumsum(L[op]) <= f * tot]].sum() / 1e6
    print(f"{disc*100:>3.0f}% | {rho:>14.4f} | {pf - rf:+.1f}")

  δ  | Spearman с δ=3% | выигрыш над risk(PD), млн $
  0% |         0.9982 | +11.7
  3% |         1.0000 | +11.5
  6% |         0.9984 | +10.9
  9% |         0.9937 | +10.2
 12% |         0.9863 | +9.9


---
**Дальше:** на этих же объектах (`money`, `profit_score`, `risk_score`, `pd36_gbm_cal`) — проверки
§3.4.1 (риск-скорректированное портфельное сравнение, Табл. 3.5), §3.5 (репликация 2014),
§3.6 (надёжность параметрической прибыли, H3), ablation источника выигрыша (Табл. 3.6),
кластерный вывод, 60-мес сплит и перебор гиперпараметров.

## Блок 6. Проверки на тех же объектах (Табл. 3.3, 3.5, 3.6, 3.8; кластерный вывод)
Все ячейки переиспользуют канонические `money`/`L,r,n_term,net,pd_money`, `profit_score`,
`risk_score`, `pv_comp` (Блок 5) и `pd36_*` (Блоки 3–4) — без переобучения моделей и без
переопределения срезов. Проверки с переобучением (репликация 2014 §3.5, PD без внутр. скоринга,
60-мес сплит, перебор гиперпараметров) — следующий, последний батч.

### 6.1 H3 (§3.6): надёжность параметрической функции прибыли (Табл. 3.8)
Параметрический вердикт «пляшет» от ненаблюдаемых LGD и m, фактический денежный поток того же
портфеля — устойчив.

In [22]:
MS = [0.10, 0.15, 0.25, 0.40, 0.60]
print('(A) ПАРАМЕТРИЧЕСКАЯ оценка прибыли одобренного портфеля, млн $:')
print(f"{'LGD/m':>6} | " + ' | '.join(f'm={d:>4}' for d in MS))
for lgd in [0.45, 0.50, 0.55]:
    row = []
    for dmf in MS:
        g, b = pv_comp(L, r, n_term, lgd=lgd, disc=0.03, dmf=dmf)
        epi = (1-pd_money)*g + pd_money*b
        row.append(f'{epi[epi>=0].sum()/1e6:>6.1f}')
    print(f'{lgd:>6} | ' + ' | '.join(row))
print('\n(B) ФАКТИЧЕСКИЙ денежный поток того же одобренного портфеля, млн $ (доля одобренных):')
print(f"{'LGD/m':>6} | " + ' | '.join(f'm={d:>4}' for d in MS))
for lgd in [0.45, 0.50, 0.55]:
    row = []
    for dmf in MS:
        g, b = pv_comp(L, r, n_term, lgd=lgd, disc=0.03, dmf=dmf)
        appr = ((1-pd_money)*g + pd_money*b) >= 0
        row.append(f'{net[appr].sum()/1e6:>6.1f} ({appr.mean()*100:>3.0f}%)')
    print(f'{lgd:>6} | ' + ' | '.join(row))

(A) ПАРАМЕТРИЧЕСКАЯ оценка прибыли одобренного портфеля, млн $:
 LGD/m | m= 0.1 | m=0.15 | m=0.25 | m= 0.4 | m= 0.6
  0.45 |  225.9 |  242.2 |  274.4 |  313.2 |  364.6
   0.5 |  206.4 |  223.7 |  257.9 |  299.5 |  355.1
  0.55 |  187.4 |  205.3 |  241.5 |  285.9 |  345.6

(B) ФАКТИЧЕСКИЙ денежный поток того же одобренного портфеля, млн $ (доля одобренных):
 LGD/m | m= 0.1 | m=0.15 | m=0.25 | m= 0.4 | m= 0.6
  0.45 |  276.7 ( 99%) |  277.2 (100%) |  277.5 (100%) |  277.5 (100%) |  277.5 (100%)
   0.5 |  275.9 ( 98%) |  276.5 ( 99%) |  277.4 (100%) |  277.5 (100%) |  277.5 (100%)
  0.55 |  273.4 ( 97%) |  275.5 ( 98%) |  277.2 (100%) |  277.5 (100%) |  277.5 (100%)


### 6.1a Устойчивость ранжирования: Спирмен/Жаккар по решётке LGD×m 
Ранговая корреляция профит-скора к базовому варианту и пересечение топ-портфелей @30% — вход для §3.6

In [23]:
# Зависимости: Блок 5 (profit_score, risk_score, net, pd_money, L, tot, pv_comp)
from scipy.stats import spearmanr
def jaccard(a, b): sa, sb = set(a.tolist()), set(b.tolist()); return len(sa & sb)/len(sa | sb)
def top(score, f): o = np.argsort(-score); return o[np.cumsum(L[o]) <= f*tot]
f = 0.30
print(f"Жаккар(профит-топ, риск-топ) @30%: {jaccard(top(profit_score,f), top(risk_score,f)):.2f}  (текст ~0.02)")
print(f"Спирмен(профит-скор, net):         {spearmanr(profit_score, net).correlation:.2f}  (текст ~0.10)")
base = profit_score; base_top = top(base, f)
print("Спирмен профит-скора к базовому и Жаккар топ-портфеля по сетке (LGD×m):")
for lgd in [0.45, 0.50, 0.55]:
    for dmf in [0.10, 0.15, 0.25, 0.40]:
        g, b = pv_comp(L, r, n_term, lgd=lgd, disc=0.03, dmf=dmf)
        s = ((1-pd_money)*g + pd_money*b)/L
        print(f"  LGD={lgd} m={int(dmf*100)}%: Спирмен={spearmanr(s, base).correlation:.3f}  "
              f"Жаккар={jaccard(top(s,f), base_top):.2f}")

Жаккар(профит-топ, риск-топ) @30%: 0.23  (текст ~0.02)
Спирмен(профит-скор, net):         0.10  (текст ~0.10)
Спирмен профит-скора к базовому и Жаккар топ-портфеля по сетке (LGD×m):
  LGD=0.45 m=10%: Спирмен=1.000  Жаккар=1.00
  LGD=0.45 m=15%: Спирмен=0.993  Жаккар=0.93
  LGD=0.45 m=25%: Спирмен=0.953  Жаккар=0.81
  LGD=0.45 m=40%: Спирмен=0.883  Жаккар=0.66
  LGD=0.5 m=10%: Спирмен=0.992  Жаккар=0.94
  LGD=0.5 m=15%: Спирмен=1.000  Жаккар=1.00
  LGD=0.5 m=25%: Спирмен=0.975  Жаккар=0.85
  LGD=0.5 m=40%: Спирмен=0.907  Жаккар=0.71
  LGD=0.55 m=10%: Спирмен=0.968  Жаккар=0.85
  LGD=0.55 m=15%: Спирмен=0.993  Жаккар=0.95
  LGD=0.55 m=25%: Спирмен=0.991  Жаккар=0.92
  LGD=0.55 m=40%: Спирмен=0.929  Жаккар=0.74


### 6.1b ДИ выигрыша по решётке LGD×m (Табл. в §3.6)
Проверка устойчивости знака выигрыша по всем 12 комбинациям (LGD ∈ {0,45; 0,50; 0,55} × m ∈ {10; 15; 25; 40 %}). Все 12 ячеек должны быть строго положительны

In [24]:
# Зависимости: Блок 5 (L,r,n_term,net,pd_money,tot,risk_score, pv_comp)
f = 0.30
def gain_ci(score, B=500, seed=SEED):
    rng = np.random.default_rng(seed)
    op = np.argsort(-score); pf = net[op[np.cumsum(L[op]) <= f*tot]].sum()/1e6
    orr = np.argsort(risk_score); rf = net[orr[np.cumsum(L[orr]) <= f*tot]].sum()/1e6
    ds = []
    for _ in range(B):
        ix = rng.integers(0, len(L), len(L)); Lb, nb_ = L[ix], net[ix]; t = Lb.sum()
        o1 = np.argsort(-score[ix]); o2 = np.argsort(risk_score[ix])
        ds.append((nb_[o1[np.cumsum(Lb[o1]) <= f*t]].sum() - nb_[o2[np.cumsum(Lb[o2]) <= f*t]].sum())/1e6)
    lo, hi = np.percentile(ds, [2.5, 97.5]); return pf-rf, lo, hi
print("Выигрыш профит над risk(PD) @30% по решётке LGD×m (95% ДИ):")
allpos = True
for lgd in [0.45, 0.50, 0.55]:
    for dmf in [0.10, 0.15, 0.25, 0.40]:
        g, b = pv_comp(L, r, n_term, lgd=lgd, disc=0.03, dmf=dmf)
        d, lo, hi = gain_ci(((1-pd_money)*g + pd_money*b)/L)
        allpos &= lo > 0
        print(f"  LGD={lgd} m={int(dmf*100):>2}%: Δ={d:+.1f} [{lo:+.1f}; {hi:+.1f}] {'+' if lo>0 else '0!'}")
print("Все 12 ячеек строго положительны:" , allpos)

Выигрыш профит над risk(PD) @30% по решётке LGD×m (95% ДИ):
  LGD=0.45 m=10%: Δ=+11.5 [+9.1; +13.8] +
  LGD=0.45 m=15%: Δ=+10.2 [+7.6; +12.6] +
  LGD=0.45 m=25%: Δ=+7.4 [+4.7; +9.7] +
  LGD=0.45 m=40%: Δ=+3.6 [+0.8; +6.0] +
  LGD=0.5 m=10%: Δ=+12.3 [+9.9; +14.4] +
  LGD=0.5 m=15%: Δ=+11.5 [+9.0; +13.7] +
  LGD=0.5 m=25%: Δ=+8.3 [+5.6; +10.6] +
  LGD=0.5 m=40%: Δ=+4.8 [+2.0; +7.1] +
  LGD=0.55 m=10%: Δ=+13.4 [+10.9; +15.4] +
  LGD=0.55 m=15%: Δ=+12.3 [+9.9; +14.5] +
  LGD=0.55 m=25%: Δ=+10.1 [+7.5; +12.3] +
  LGD=0.55 m=40%: Δ=+5.7 [+3.0; +8.2] +
Все 12 ячеек строго положительны: True


### 6.2 Декомпозиция S0/S1/S2 (Табл. 3.3)
S0 — логрег + порог по F1; S1 — бустинг + порог по F1; S2 — бустинг + порог по прибыли (одобрить всех).

In [25]:
# 6.2 Декомпозиция S0/S1/S2 (Табл. 3.3): корректная версия.
# Здесь же определяются helper'ы, нужные для всего Блока 6.2:
# pd36_lr_valid (LR-PD@36 на валидации 2014), mat_mask, best_f1_thr.
from sklearn.metrics import f1_score

mat_mask = money['mask']                       # дозревшая денежная маска теста 2015

# LR-hazard PD@36 на валидации 2014 (GBM-версия pd36_gbm_valid уже есть из Блока 4)
pd36_lr_valid = survival_pd36(
    lambda X: lr_hazard.predict_proba(sc_hazard.transform(imp_hazard.transform(X)))[:, 1],
    surv_valid)

def best_f1_thr(y, p):
    """F1-оптимальный порог по сетке вероятностей [0.02; 0.95]."""
    grid = np.linspace(0.02, 0.95, 200)
    return grid[int(np.argmax([f1_score(y, (p >= t).astype(int)) for t in grid]))]

pd36_gbm_cal_valid = iso_calibrator.transform(pd36_gbm_valid)     # калиброванная валидация GBM
thr_lr  = best_f1_thr(y36_valid, pd36_lr_valid)                   # LR: некалибр. шкала
thr_gbm = best_f1_thr(y36_valid, pd36_gbm_cal_valid)              # GBM: КАЛИБР. шкала

S0 = net[pd36_lr[mat_mask]      < thr_lr ].sum()/1e6              # логрег + F1
S1 = net[pd36_gbm_cal[mat_mask] < thr_gbm].sum()/1e6              # бустинг + F1 на калибр. шкале
S2 = net.sum()/1e6                                                # бустинг + порог по прибыли = одобрить всех

appr_lr  = (pd36_lr[mat_mask]      < thr_lr ).mean()
appr_gbm = (pd36_gbm_cal[mat_mask] < thr_gbm).mean()

print(f'S0 (логрег + F1):              {S0:>+8.2f} млн')
print(f'S1 (бустинг + F1, калибр.):    {S1:>+8.2f} млн   Δ_model     = {S1-S0:+.2f}')
print(f'S2 (бустинг + порог-прибыль):  {S2:>+8.2f} млн   Δ_threshold = {S2-S1:+.2f}')
print(f'Доля одобрения: LR={appr_lr*100:.0f}%  GBM={appr_gbm*100:.0f}%   (текст §3.3: 72% / 76%)')

S0 (логрег + F1):               +218.01 млн
S1 (бустинг + F1, калибр.):     +230.74 млн   Δ_model     = +12.73
S2 (бустинг + порог-прибыль):   +277.53 млн   Δ_threshold = +46.79
Доля одобрения: LR=72%  GBM=76%   (текст §3.3: 72% / 76%)


### 6.2a Вклад модели (GBM vs LR) в профит-портфель

Сравнение профит-портфелей на GBM-PD и калиброванной LR-PD. 

In [26]:
# 6.2a Вклад модели (GBM vs LR) в профит-портфель.
# Зависимости: Блок 4 (iso_calibrator, pd36_lr, y36_valid), 6.2 (pd36_lr_valid),
# Блок 5 (money, L, r, n_term, net, tot, profit_score, pv_comp).
iso_lr = IsotonicRegression(out_of_bounds='clip').fit(pd36_lr_valid, y36_valid)
pd_lr_money = iso_lr.transform(pd36_lr)[money['mask']]
g, b = pv_comp(L, r, n_term, lgd=0.50, disc=0.03)
profit_lr = ((1-pd_lr_money)*g + pd_lr_money*b) / L          # профит-скор на LR-PD

def jaccard(a, b): sa, sb = set(a.tolist()), set(b.tolist()); return len(sa & sb)/len(sa | sb)
def sel(score, f): o = np.argsort(-score); return o[np.cumsum(L[o]) <= f*tot]

rng = np.random.default_rng(SEED)
for f in [0.10, 0.30]:
    g_gbm = net[sel(profit_score, f)].sum()/1e6; g_lr = net[sel(profit_lr, f)].sum()/1e6
    ds = []
    for _ in range(2000):
        ix = rng.integers(0, len(L), len(L)); Lb, nb_ = L[ix], net[ix]; t = Lb.sum()
        og = np.argsort(-profit_score[ix]); ol = np.argsort(-profit_lr[ix])
        ds.append((nb_[og[np.cumsum(Lb[og]) <= f*t]].sum() - nb_[ol[np.cumsum(Lb[ol]) <= f*t]].sum())/1e6)
    lo, hi = np.percentile(ds, [2.5, 97.5])
    print(f'{int(f*100)}%: вклад модели (GBM−LR) = {g_gbm-g_lr:+.1f} млн  95%ДИ[{lo:+.1f}; {hi:+.1f}]  '
          f'Жаккар(GBM,LR)={jaccard(sel(profit_score,f), sel(profit_lr,f)):.2f}')
    

10%: вклад модели (GBM−LR) = +2.6 млн  95%ДИ[+1.4; +3.8]  Жаккар(GBM,LR)=0.49
30%: вклад модели (GBM−LR) = +3.1 млн  95%ДИ[+1.6; +4.4]  Жаккар(GBM,LR)=0.63


In [27]:
# Подбор F1-порога ДЛЯ КАЖДОЙ модели на ЕЁ ЖЕ шкале (LR — некалиброванная, GBM — калиброванная).
pd36_gbm_cal_valid = iso_calibrator.transform(pd36_gbm_valid)     # калиброванная валидация GBM
thr_lr  = best_f1_thr(y36_valid, pd36_lr_valid)                   # LR: некалибр. шкала (как в S0/тесте)
thr_gbm = best_f1_thr(y36_valid, pd36_gbm_cal_valid)              # GBM: КАЛИБР. шкала (как в S1/тесте)
S0 = net[pd36_lr[mat_mask]      < thr_lr ].sum()/1e6
S1 = net[pd36_gbm_cal[mat_mask] < thr_gbm].sum()/1e6
S2 = net.sum()/1e6
appr_lr  = (pd36_lr[mat_mask]      < thr_lr ).mean()
appr_gbm = (pd36_gbm_cal[mat_mask] < thr_gbm).mean()
print(f'S0 {S0:+.2f} | S1 {S1:+.2f} (Δ_model {S1-S0:+.2f}) | S2 {S2:+.2f} (Δ_thr {S2-S1:+.2f})')
print(f'Доля одобрения: LR={appr_lr*100:.0f}%  GBM={appr_gbm*100:.0f}%   (текст §3.3: 72% / 76%)')

S0 +218.01 | S1 +230.74 (Δ_model +12.73) | S2 +277.53 (Δ_thr +46.79)
Доля одобрения: LR=72%  GBM=76%   (текст §3.3: 72% / 76%)


### 6.3 Источник выигрыша H2 (Табл. 3.6)
Само добавление ставки выигрыша не даёт; значимый выигрыш — только у полной формы прибыли.
Плюс разложение: какая доля выигрыга — от крена к высокому APR, какая — от отбора внутри APR-уровней.

In [28]:
g, b = pv_comp(L, r, n_term, lgd=0.50, disc=0.03)
i_m = r/12.; A_m = np.where(i_m>0, L*i_m/(1-(1+i_m)**(-n_term)), L/n_term)
exp_int_per_dollar = ((A_m*n_term - L)/L) * (1-pd_money)               # ожидаемый процентный доход на $
g_nr, b_nr = pv_comp(L, np.full_like(r, r.mean()), n_term, lgd=0.50, disc=0.03)  # та же форма, ставка=средней
abl = {
    'profit/$ (база)':   -((1-pd_money)*g + pd_money*b)/L,
    'risk(PD)':           pd_money,
    'rate':              -r,
    'rate×(1−PD)':       -(r*(1-pd_money)),
    'E[%дохода]/$':      -exp_int_per_dollar,
    'profit_без_ставки': -((1-pd_money)*g_nr + pd_money*b_nr)/L,
}
abl_orders = {k: np.argsort(v) for k, v in abl.items()}
abl_orders['random'] = np.random.default_rng(SEED).permutation(len(L))
def _port(o, f): return o[np.cumsum(L[o]) <= f*tot]
print(f"{'бюджет':>7} | " + ' | '.join(f'{k:>16}' for k in abl_orders))
for f in [0.10, 0.30]:
    print(f'{int(f*100):>6}% | ' + ' | '.join(f'{net[_port(o, f)].sum()/1e6:>16.1f}' for o in abl_orders.values()))

# Разложение выигрыша: крен по APR vs отбор сверх APR-профиля (бюджет 30%)
f = 0.30
prof_idx = _port(np.argsort(-profit_score), f); risk_idx = _port(np.argsort(risk_score), f)
prof_flow = net[prof_idx].sum()/1e6; risk_flow = net[risk_idx].sum()/1e6
NB = 20
edges = np.quantile(r, np.linspace(0, 1, NB+1)); edges[0] = -np.inf; edges[-1] = np.inf
bin_all = np.clip(np.digitize(r, edges[1:-1]), 0, NB-1); bin_tg = bin_all[prof_idx]
def apr_matched_random(B=300):
    vals = []
    for seed in range(B):
        rr = np.random.default_rng(seed); sel = []
        for bb in range(NB):
            tgtL = L[prof_idx[bin_tg == bb]].sum()
            if tgtL <= 0: continue
            pool = rr.permutation(np.where(bin_all == bb)[0]); c = np.cumsum(L[pool]); sel.append(pool[c <= tgtL])
        idx = np.concatenate(sel) if sel else np.array([], int); vals.append(net[idx].sum()/1e6)
    return float(np.mean(vals))
apr_rand = apr_matched_random()
gain_total = prof_flow - risk_flow; gain_apr = apr_rand - risk_flow; gain_beyond = prof_flow - apr_rand
print(f'\nРазложение выигрыша profit − risk(PD) @30% (всего {gain_total:+.1f} млн):')
print(f'  крен по APR (APR-matched random): {gain_apr:+.1f} млн ({gain_apr/gain_total*100:.0f}%)')
print(f'  отбор сверх APR-профиля:          {gain_beyond:+.1f} млн ({gain_beyond/gain_total*100:.0f}%)')

 бюджет |  profit/$ (база) |         risk(PD) |             rate |      rate×(1−PD) |     E[%дохода]/$ | profit_без_ставки |           random
    10% |             33.6 |             26.3 |             21.6 |             26.9 |             26.6 |             26.3 |             26.8
    30% |             93.9 |             82.3 |             79.3 |             83.5 |             83.1 |             82.4 |             81.3

Разложение выигрыша profit − risk(PD) @30% (всего +11.5 млн):
  крен по APR (APR-matched random): +1.4 млн (12%)
  отбор сверх APR-профиля:          +10.1 млн (88%)


### 6.3a Lowest-PD внутри APR-профиля (§3.4.2)
Контрольный отбор: внутри каждого APR-страта берём кредиты с минимальной PD. Должен почти точно воспроизвести профит-портфель (~94 млн), подтверждая, что выигрыш = скрининг по PD внутри APR-уровней

In [29]:
# Зависимости: Блок 5/6 (L,r,net,pd_money,profit_score,tot)
f = 0.30
prof_idx = np.argsort(-profit_score); prof_idx = prof_idx[np.cumsum(L[prof_idx]) <= f*tot]
NB = 20; edges = np.quantile(r, np.linspace(0, 1, NB+1)); edges[0] = -np.inf; edges[-1] = np.inf
bin_all = np.clip(np.digitize(r, edges[1:-1]), 0, NB-1); bin_tg = bin_all[prof_idx]
sel = []
for bb in range(NB):
    tgtL = L[prof_idx[bin_tg == bb]].sum()
    if tgtL <= 0: continue
    pool = np.where(bin_all == bb)[0]; pool = pool[np.argsort(pd_money[pool])]   # минимальная PD внутри страта
    sel.append(pool[np.cumsum(L[pool]) <= tgtL])
idx = np.concatenate(sel)
print(f"lowest-PD внутри APR-профиля: {net[idx].sum()/1e6:.1f} млн (профит-портфель {net[prof_idx].sum()/1e6:.1f})")
for name, order in [('крупные первыми', np.argsort(-L)), ('мелкие первыми', np.argsort(L))]:
    s = order[np.cumsum(L[order]) <= f*tot]; print(f"  {name}: {net[s].sum()/1e6:.1f} млн")

lowest-PD внутри APR-профиля: 94.0 млн (профит-портфель 93.9)
  крупные первыми: 88.5 млн
  мелкие первыми: 82.7 млн


### 6.3b Ablation с доверительными интервалами (Табл. 3.6)
Те же правила, что в 6.3, но с бутстрэп-ДИ для разницы с risk(PD). Полная форма прибыли — единственное правило со значимым выигрышем

In [30]:
# Зависимости: ячейка 6.3 должна быть выполнена (нужен словарь abl со ЗНАКОМ под argsort-ascending)
f = 0.30
def rule_ci(score, B=500, seed=SEED):                 # score уже со знаком: argsort ascending = лучшие первыми
    rng = np.random.default_rng(seed)
    op = np.argsort(score); orr = np.argsort(risk_score)
    pf = net[op[np.cumsum(L[op]) <= f*tot]].sum()/1e6; rf = net[orr[np.cumsum(L[orr]) <= f*tot]].sum()/1e6
    ds = []
    for _ in range(B):
        ix = rng.integers(0, len(L), len(L)); Lb, nb_ = L[ix], net[ix]; t = Lb.sum()
        o1 = np.argsort(score[ix]); o2 = np.argsort(risk_score[ix])
        ds.append((nb_[o1[np.cumsum(Lb[o1]) <= f*t]].sum() - nb_[o2[np.cumsum(Lb[o2]) <= f*t]].sum())/1e6)
    lo, hi = np.percentile(ds, [2.5, 97.5]); return pf-rf, lo, hi
print("Выигрыш над risk(PD) @30% по правилам ablation (95% ДИ):")
for k, v in abl.items():
    if k == 'risk(PD)': continue
    d, lo, hi = rule_ci(v)
    print(f"  {k:>18}: Δ={d:+.1f} [{lo:+.1f}; {hi:+.1f}] {'значимо' if lo>0 else ('значимо хуже' if hi<0 else 'неотличимо')}")

Выигрыш над risk(PD) @30% по правилам ablation (95% ДИ):
     profit/$ (база): Δ=+11.5 [+9.0; +13.7] значимо
                rate: Δ=-3.1 [-6.0; -0.6] значимо хуже
         rate×(1−PD): Δ=+1.1 [-1.8; +3.6] неотличимо
        E[%дохода]/$: Δ=+0.7 [-2.1; +3.4] неотличимо
   profit_без_ставки: Δ=+0.1 [-0.1; +0.2] неотличимо


### 6.3c Разложение по APR-квинтилям (Табл. 3.6a / 3.6b)

In [31]:
# === 6.3a Разложение механизма H2 по APR-квинтилям (тест 2015, бюджет 30%, δ=3%) ===
# Замечание №4: на платформе PD скоррелирована с APR (риск-ценообразование),
# поэтому отбор по низкой PD концентрируется в Q1–Q2 (низкая ставка → низкий поток),
# а профит-отбор — в Q4–Q5, где доход компенсирует риск.

NQ = 5
edges_apr = np.quantile(r, np.linspace(0, 1, NQ+1))
edges_apr[0] = -np.inf; edges_apr[-1] = np.inf
apr_bin = np.clip(np.digitize(r, edges_apr[1:-1]), 0, NQ-1)

f = 0.30
sel = {
    'profit'  : orders['profit']  [np.cumsum(L[orders['profit']])   <= f*tot],
    'risk(PD)': orders['risk(PD)'][np.cumsum(L[orders['risk(PD)']]) <= f*tot],
    'random'  : orders['random']  [np.cumsum(L[orders['random']])   <= f*tot],
}

# --- Таблица A: характеристики APR-квинтилей во всей дозревшей денежной выборке ---
print('Таблица A. Структура популяции по APR-квинтилям (дозревшая выборка теста 2015)')
print(f"{'квинт.':>7} | {'APR, %':>13} | {'n':>9} | {'ср.PD@36':>9} | "
      f"{'дефолт@36':>10} | {'ср. net/$, %':>13}")
print('-' * 85)
for q in range(NQ):
    m = (apr_bin == q)
    print(f"     Q{q+1} | {r[m].min()*100:5.1f}–{r[m].max()*100:>5.1f}  | "
          f"{m.sum():>9,} | {pd_money[m].mean()*100:>8.1f}% | "
          f"{yv_money[m].mean()*100:>9.1f}% | {(net[m]/L[m]).mean()*100:>+12.1f}%")

# --- Таблица B: аллокация капитала и реализованный поток при бюджете 30 % ---
print(f'\nТаблица B. Аллокация капитала и поток по APR-квинтилям, бюджет {int(f*100)} %')
print(f"{'квинт.':>7} | {'profit: %кап / поток':>22} | "
      f"{'risk(PD): %кап / поток':>24} | {'random: %кап / поток':>22}   (поток, млн $)")
print('-' * 110)
for q in range(NQ):
    cells = []
    for k in ['profit', 'risk(PD)', 'random']:
        s = sel[k]; ms = s[apr_bin[s] == q]
        cap_share = L[ms].sum() / L[s].sum() * 100
        flow_q    = net[ms].sum() / 1e6
        cells.append(f"{cap_share:>7.1f}% / {flow_q:>+7.1f}")
    print(f"     Q{q+1} | {cells[0]:>22} | {cells[1]:>24} | {cells[2]:>22}")
print('-' * 110)
sums = {k: net[sel[k]].sum()/1e6 for k in sel}
profit_total = f"100.0% / {sums['profit']:+5.1f}"
risk_total   = f"100.0% / {sums['risk(PD)']:+5.1f}"
random_total = f"100.0% / {sums['random']:+5.1f}"

print(
    f"   итого | {profit_total:>22} | "
    f"{risk_total:>24} | "
    f"{random_total:>22}"
)

Таблица A. Структура популяции по APR-квинтилям (дозревшая выборка теста 2015)
 квинт. |        APR, % |         n |  ср.PD@36 |  дефолт@36 |  ср. net/$, %
-------------------------------------------------------------------------------------
     Q1 |   5.3–  7.5  |    49,787 |      4.4% |       4.5% |         +6.9%
     Q2 |   7.9–  9.8  |    58,561 |      8.2% |       9.1% |         +7.6%
     Q3 |  10.0– 12.1  |    57,232 |     11.2% |      13.6% |         +8.4%
     Q4 |  12.3– 14.0  |    60,579 |     15.7% |      18.9% |         +8.1%
     Q5 |  14.3– 29.0  |    56,867 |     22.3% |      26.9% |         +7.3%

Таблица B. Аллокация капитала и поток по APR-квинтилям, бюджет 30 %
 квинт. |   profit: %кап / поток |   risk(PD): %кап / поток |   random: %кап / поток   (поток, млн $)
--------------------------------------------------------------------------------------------------------------
     Q1 |         0.0% /    +0.0 |          61.9% /   +47.6 |        19.7% /   +14.7
     Q2 |  

### 6.4 Риск-скорректированное портфельное сравнение (Табл. 3.5)
Корректный объект риска — распределение доходности портфеля на капитал (ROC), с разделением
диверсифицируемой (i.i.d.) и недиверсифицируемой (винтажной, кластерный бутстрэп по месяцу выдачи)
частей. Пер-loan VaR/ES приведены лишь как контраст (в портфеле диверсифицируются).

In [32]:
month_money = surv_test['issue_date'].dt.month.values[money['mask']]
ra_orders = {'profit': np.argsort(-profit_score), 'risk(PD)': np.argsort(risk_score)}
def ra_port(o, f): return o[np.cumsum(L[o]) <= f*tot]
rng = np.random.default_rng(SEED)
groups_all = {mo: np.where(month_money == mo)[0] for mo in np.unique(month_money)}
B = 2000

def risk_block(sel):
    x = net[sel]; Ls = L[sel]; nn = len(x); cap = Ls.sum(); roc_pt = x.sum()/cap
    sd_loan = x.std(); var5_loan = np.percentile(x, 5); es5_loan = x[x <= var5_loan].mean()
    roc_iid = np.empty(B)
    for j in range(B):
        ix = rng.integers(0, nn, nn); roc_iid[j] = x[ix].sum()/Ls[ix].sum()
    sd_iid = roc_iid.std()
    sel_set = set(sel.tolist())
    gsel = {mo: np.array([i for i in idx if i in sel_set]) for mo, idx in groups_all.items()}
    gsel = {mo: v for mo, v in gsel.items() if len(v) > 0}; keys = list(gsel)
    roc_clu = np.empty(B)
    for j in range(B):
        pick = rng.choice(keys, size=len(keys), replace=True)
        ii = np.concatenate([gsel[mo] for mo in pick]); roc_clu[j] = net[ii].sum()/L[ii].sum()
    sd_clu = roc_clu.std(); var5_clu = np.percentile(roc_clu, 5); es5_clu = roc_clu[roc_clu <= var5_clu].mean()
    return dict(n=nn, sumM=x.sum()/1e6, roc=roc_pt*100, sd_loan=sd_loan,
                var5_loan=var5_loan, es5_loan=es5_loan, sd_iid=sd_iid*100, sd_clu=sd_clu*100,
                var5_clu=var5_clu*100, es5_clu=es5_clu*100, ratio=sd_clu/sd_iid, p_neg=(roc_clu<0).mean()*100)

for f in [0.10, 0.30]:
    sp = risk_block(ra_port(ra_orders['profit'], f)); sr = risk_block(ra_port(ra_orders['risk(PD)'], f))
    print(f'\n========== БЮДЖЕТ {int(f*100)}% ==========')
    print(f"{'показатель':>34} | {'profit':>10} | {'risk(PD)':>10}")
    for lbl, key, fmt in [('поток, млн $','sumM','{:+.1f}'), ('доходность на капитал, %','roc','{:+.2f}'),
        ('σ(net)/кред, $ (пер-loan)','sd_loan','{:,.0f}'), ('VaR5%/кред, $ (пер-loan)','var5_loan','{:+,.0f}'),
        ('σ(ROC) i.i.d., п.п.','sd_iid','{:.3f}'), ('σ(ROC) кластерн., п.п.','sd_clu','{:.3f}'),
        ('VaR5% ROC кластерн., %','var5_clu','{:+.2f}'), ('ES5% ROC кластерн., %','es5_clu','{:+.2f}'),
        ('P(ROC<0) кластерн., %','p_neg','{:.1f}'), ('σ_кластер/σ_iid','ratio','{:.1f}')]:
        print(f'{lbl:>34} | {fmt.format(sp[key]):>10} | {fmt.format(sr[key]):>10}')


========== БЮДЖЕТ 10% ==========
                        показатель |     profit |   risk(PD)
                      поток, млн $ |      +33.6 |      +26.3
          доходность на капитал, % |      +9.27 |      +7.26
         σ(net)/кред, $ (пер-loan) |      4,318 |      1,632
          VaR5%/кред, $ (пер-loan) |     -5,962 |       +170
               σ(ROC) i.i.d., п.п. |      0.195 |      0.063
            σ(ROC) кластерн., п.п. |      0.355 |      0.084
            VaR5% ROC кластерн., % |      +8.67 |      +7.13
             ES5% ROC кластерн., % |      +8.52 |      +7.10
             P(ROC<0) кластерн., % |        0.0 |        0.0
                   σ_кластер/σ_iid |        1.8 |        1.3

========== БЮДЖЕТ 30% ==========
                        показатель |     profit |   risk(PD)
                      поток, млн $ |      +93.9 |      +82.3
          доходность на капитал, % |      +8.63 |      +7.57
         σ(net)/кред, $ (пер-loan) |      3,929 |      2,034
          VaR5%/к

### 6.4a Доп. диагностика: ДИ AUC, ES5%/кред, reliability diagram

Три проверки, дополняющие основной анализ:
(1) бутстрэп-ДИ абсолютного AUC для GBM- и LR-hazard;
(2) ES 5 % на отдельный кредит для профит- и риск-портфеля @30 %
(в портфеле эти потери диверсифицируются, см. §3.4.3);
(3) калибровочная диаграмма по 10 бинам для GBM-cal — вход для рисунка
reliability в §3.2.

In [33]:
# Зависимости: Блок 3 (auc_*_hazard, pd36_*, y36_test), Блок 4 (pd36_gbm_cal, ece),
# Блок 5 (net, L, tot, profit_score, risk_score).
# (1) Абсолютные ДИ AUC по моделям
rng = np.random.default_rng(SEED); N = len(y36_test); ag, al = [], []
for _ in range(2000):
    ix = rng.integers(0, N, N)
    ag.append(roc_auc_score(y36_test[ix], pd36_gbm[ix])); al.append(roc_auc_score(y36_test[ix], pd36_lr[ix]))
print(f"AUC GBM-hazard = {auc_gbm_hazard:.4f} 95%ДИ[{np.percentile(ag,2.5):.4f}; {np.percentile(ag,97.5):.4f}]")
print(f"AUC LR-hazard  = {auc_lr_hazard:.4f} 95%ДИ[{np.percentile(al,2.5):.4f}; {np.percentile(al,97.5):.4f}]")

# (2) ES 5 % на кредит (то, что Табл. 3.5 показывает, но 6.4 не печатает)
def es5_loan(idx): x = net[idx]; v = np.percentile(x, 5); return x[x <= v].mean()
f = 0.30
for nm, sc in [('profit', -profit_score), ('risk(PD)', risk_score)]:
    o = np.argsort(sc); s = o[np.cumsum(L[o]) <= f*tot]
    print(f"ES5%/кред {nm}: ${es5_loan(s):+,.0f}")

# (3) Reliability diagram (10 бинов): прогноз vs факт
edges = np.linspace(0, 1, 11)
print("bin | n | ср.прогноз | факт")
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (pd36_gbm_cal >= lo) & ((pd36_gbm_cal < hi) if hi < 1 else (pd36_gbm_cal <= hi))
    if m.sum(): print(f"[{lo:.1f};{hi:.1f}) | {int(m.sum()):>7,} | {pd36_gbm_cal[m].mean():.3f} | {y36_test[m].mean():.3f}")
# Для рисунка: plt.plot(pred, obs) + диагональ y=x

AUC GBM-hazard = 0.6900 95%ДИ[0.6879; 0.6919]
AUC LR-hazard  = 0.6875 95%ДИ[0.6855; 0.6896]
ES5%/кред profit: $-11,356
ES5%/кред risk(PD): $-4,960
bin | n | ср.прогноз | факт
[0.0;0.1) | 153,747 | 0.065 | 0.079
[0.1;0.2) | 168,247 | 0.148 | 0.176
[0.2;0.3) |  66,313 | 0.237 | 0.281
[0.3;0.4) |  32,284 | 0.332 | 0.378
[0.4;0.5) |     502 | 0.444 | 0.488
[0.9;1.0) |       2 | 1.000 | 0.000


### 6.5 Инференция выигрыша при 12 кластерах: cluster-robust t(11) + wild cluster bootstrap

In [34]:
from scipy.stats import t as tdist
month_c = surv_test['issue_date'].dt.month.values[money['mask']]
uniq, inv = np.unique(month_c, return_inverse=True); G = len(uniq); ng = np.bincount(inv).astype(float)
def c_port(o, f): return o[np.cumsum(L[o]) <= f*tot]
print(f'Кластеров (месяцев выдачи): {G}')
rng = np.random.default_rng(SEED); Bw = 99999
for f in [0.10, 0.30]:
    Ip = np.zeros(len(L), bool); Ip[c_port(np.argsort(-profit_score), f)] = True
    Ir = np.zeros(len(L), bool); Ir[c_port(np.argsort(risk_score), f)] = True
    d = net*(Ip.astype(float) - Ir.astype(float)); Nn = len(d)
    Dg = np.bincount(inv, weights=d, minlength=G); beta = d.sum()/Nn
    Sg = Dg - beta*ng; SEbeta = np.sqrt((Sg**2).sum()/Nn**2); t_obs = beta/SEbeta
    G_M = d.sum()/1e6; hw = tdist.ppf(0.975, G-1)*SEbeta*Nn/1e6
    W = rng.choice([-1.0, 1.0], size=(Bw, G)); betastar = (W@Dg)/Nn
    Sgstar = W*Dg[None, :] - betastar[:, None]*ng[None, :]; Vstar = (Sgstar**2).sum(1)/Nn**2
    tstar = betastar/np.sqrt(Vstar); p = (1 + np.sum(np.abs(tstar) >= abs(t_obs)))/(Bw+1)
    print(f'\n--- бюджет {int(f*100)}% ---')
    print(f'  выигрыш = {G_M:+.1f} млн | cluster-robust t({G-1}) = {t_obs:.2f}')
    print(f'  CR t({G-1}) 95% ДИ: [{G_M-hw:+.1f}; {G_M+hw:+.1f}] млн')
    print(f'  wild cluster bootstrap p-value (H0: выигрыш=0): {p:.5f}')

Кластеров (месяцев выдачи): 12

--- бюджет 10% ---
  выигрыш = +7.3 млн | cluster-robust t(11) = 2.98
  CR t(11) 95% ДИ: [+1.9; +12.6] млн
  wild cluster bootstrap p-value (H0: выигрыш=0): 0.01766

--- бюджет 30% ---
  выигрыш = +11.5 млн | cluster-robust t(11) = 3.00
  CR t(11) 95% ДИ: [+3.1; +19.9] млн
  wild cluster bootstrap p-value (H0: выигрыш=0): 0.01657


### 6.5a Перцентильный кластерный бутстрэп по месяцу выдачи 
Альтернативный способ оценки кластерной неопределённости (12 месячных когорт). Точечная оценка совпадает с 6.5, интервал шире, но знак сохраняется

In [35]:
# Зависимости: Блок 5/6 (L,net,profit_score,risk_score,tot, money, surv_test)
month_c = surv_test['issue_date'].dt.month.values[money['mask']]
uniq = np.unique(month_c); groups = {mo: np.where(month_c == mo)[0] for mo in uniq}
def gain_on(idx, f):
    Lb, nb_, ps, rs = L[idx], net[idx], profit_score[idx], risk_score[idx]; t = Lb.sum()
    op = np.argsort(-ps); orr = np.argsort(rs)
    return (nb_[op[np.cumsum(Lb[op]) <= f*t]].sum() - nb_[orr[np.cumsum(Lb[orr]) <= f*t]].sum())/1e6
rng = np.random.default_rng(SEED)
for f in [0.10, 0.30]:
    ds = []
    for _ in range(2000):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        ds.append(gain_on(np.concatenate([groups[mo] for mo in pick]), f))
    print(f'{int(f*100)}%: перцентильный кластерный 95%ДИ [{np.percentile(ds,2.5):+.1f}; {np.percentile(ds,97.5):+.1f}] млн')

10%: перцентильный кластерный 95%ДИ [+5.3; +9.1] млн
30%: перцентильный кластерный 95%ДИ [+7.7; +15.0] млн


---
**Последний батч (с переобучением моделей):** репликация на 2014 (§3.5, Табл. 3.7),
PD без int_rate/grade/sub_grade (§3.4), H2 на 60-мес чистым сплитом train2012/test2013,
перебор гиперпараметров GBM. Эти проверки обучают отдельные модели, поэтому идут отдельным блоком.

## Блок 7. Проверки с переобучением моделей (§3.5, §3.4)
Каждая ячейка обучает отдельную модель, поэтому вынесена в отдельный блок (несколько минут на прогон).
Переиспользуются помощники `build_loan_matrix`, `expand_person_period`, `pv_comp`, `y_default_within_H`,
`RES_OK`, `hazard_cols` и канонические `money`/`L,r,n_term,net,tot`.

In [36]:
# Общий помощник: PD@36 модели с ПРОИЗВОЛЬНЫМ набором опорных колонок
def survival_pd_cols(predict_fn, df, cols):
    Xl = build_loan_matrix(df, cols).astype(np.float32).values; nn = len(df); sv = np.ones(nn)
    for t in range(1, H+1):
        sv *= (1 - predict_fn(np.column_stack([Xl, np.full(nn, t, np.float32)])))
    return 1 - sv

### 7.1 Репликация H2 на втором OOT-периоде (§3.5, Табл. 3.7)
Обучение 2012, валидация 2013, тест 2014 — независимая проверка знака и значимости H2.

In [37]:
def run_period(tr_years, va_year, te_year, disc=0.03):
    trd = surv[surv.issue_year.isin(tr_years)]; vad = surv[surv.issue_year == va_year]; ted = surv[surv.issue_year == te_year]
    Xtr, ytr, wtr = expand_person_period(trd, hazard_cols)
    Xva, yva, wva = expand_person_period(vad, hazard_cols)
    mdl = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
        min_child_samples=200, random_state=SEED, verbose=-1)
    mdl.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xva, yva)], eval_sample_weight=[wva],
            callbacks=[lgb.early_stopping(50, verbose=False)])
    pred = lambda X: mdl.predict_proba(X)[:, 1]
    cal = IsotonicRegression(out_of_bounds='clip').fit(survival_pd_cols(pred, vad, hazard_cols), y_default_within_H(vad))
    pd_te = cal.transform(survival_pd_cols(pred, ted, hazard_cols))
    mob = ted['mob'].values; term = ted['term_months'].values
    msk = (mob >= term) & ted['loan_status'].isin(RES_OK).values
    Lp = ted['loan_amnt'].values[msk].astype(float); rp = ted['int_rate'].values[msk]/100
    npp = ted['term_months'].values[msk].astype(float)
    netp = ted['total_pymnt'].values[msk] + ted['recoveries'].fillna(0).values[msk] - Lp
    pdc = pd_te[msk]; yv = ((ted['event'].values[msk] == 1) & (ted['duration'].values[msk] <= H)).astype(int)
    g, b = pv_comp(Lp, rp, npp, lgd=0.50, disc=disc)
    profit = -((1-pdc)*g + pdc*b)/Lp; risk = pdc; tot_p = Lp.sum(); rng = np.random.default_rng(SEED)
    out = {'test': te_year, 'n': int(msk.sum()), 'auc': roc_auc_score(yv, pdc)}
    for f in [0.10, 0.30, 0.50]:
        op = np.argsort(profit); orr = np.argsort(risk)
        bp = netp[op[np.cumsum(Lp[op]) <= f*tot_p]].sum()/1e6
        br = netp[orr[np.cumsum(Lp[orr]) <= f*tot_p]].sum()/1e6
        ds = []
        for _ in range(2000):
            ix = rng.integers(0, len(Lp), len(Lp)); Lb, nb_ = Lp[ix], netp[ix]; t = Lb.sum()
            o1 = np.argsort(profit[ix]); o2 = np.argsort(risk[ix])
            ds.append((nb_[o1[np.cumsum(Lb[o1]) <= f*t]].sum() - nb_[o2[np.cumsum(Lb[o2]) <= f*t]].sum())/1e6)
        lo, hi = np.percentile(ds, [2.5, 97.5]); out[f] = (bp, br, bp-br, lo, hi)
    return out

for years in ([2012, 2013], 2014, 2015), ([2012], 2013, 2014):
    rr = run_period(*years)
    print(f"\n=== тест {rr['test']}: n={rr['n']:,}, AUC@36={rr['auc']:.4f} ===")
    for f in [0.10, 0.30, 0.50]:
        bp, br, d, lo, hi = rr[f]
        print(f"  {int(f*100)}%: profit={bp:+.1f} risk={br:+.1f} Δ={d:+.1f}млн "
              f"CI[{lo:+.1f}; {hi:+.1f}] {'значимо' if lo>0 else '—'}")


=== тест 2015: n=283,026, AUC@36=0.6837 ===
  10%: profit=+33.6 risk=+26.3 Δ=+7.3млн CI[+5.8; +8.7] значимо
  30%: profit=+93.9 risk=+82.3 Δ=+11.5млн CI[+9.0; +13.8] значимо
  50%: profit=+149.6 risk=+143.8 Δ=+5.7млн CI[+3.3; +8.5] значимо

=== тест 2014: n=162,570, AUC@36=0.6663 ===
  10%: profit=+26.9 risk=+16.4 Δ=+10.5млн CI[+9.4; +11.5] значимо
  30%: profit=+74.6 risk=+55.5 Δ=+19.1млн CI[+17.6; +20.8] значимо
  50%: profit=+117.7 risk=+99.7 Δ=+18.0млн CI[+16.3; +19.9] значимо


### 7.2 PD без int_rate / grade / sub_grade (§3.4)
Контроль эндогенности: убираем из признаков выходы внутреннего скоринга платформы (ставка остаётся
в функции прибыли как контрактный параметр).

In [38]:
NUM_R = [c for c in NUM_FEATURES if c != 'int_rate']
CAT_R = [c for c in CAT_FEATURES if c not in ('grade', 'sub_grade')]
print('Исключены из признаков:', sorted(set(NUM_FEATURES+CAT_FEATURES) - set(NUM_R+CAT_R)))

def build_loan_matrix_r(df, base_cols):
    X = pd.concat([df[NUM_R].reset_index(drop=True),
                   pd.get_dummies(df[CAT_R].astype(str).reset_index(drop=True), drop_first=True)], axis=1)
    return X if base_cols is None else X.reindex(columns=base_cols, fill_value=0)

def expand_r(df, base_cols, q=0.15, seed=SEED):
    df = df.reset_index(drop=True); dur = df['duration'].values
    ev_H = ((df['event'].values == 1) & (dur <= H)).astype(int)
    T = np.clip(np.minimum(dur, H).astype(int), 1, H)
    idx = np.repeat(np.arange(len(df)), T)
    mob = np.ones(int(T.sum()), int); mob[np.cumsum(T)[:-1]] -= T[:-1]; mob = np.cumsum(mob)
    y = ((mob == np.repeat(T, T)) & (np.repeat(ev_H, T) == 1)).astype(np.int8)
    rng_e = np.random.default_rng(seed); keep = (y == 1) | (rng_e.random(len(y)) < q)
    w = np.where(y[keep] == 1, 1.0, 1.0/q).astype(np.float32)
    Xl = build_loan_matrix_r(df, base_cols).astype(np.float32).values
    return np.column_stack([Xl[idx[keep]], mob[keep].astype(np.float32)]), y[keep], w

cols_r = list(build_loan_matrix_r(surv_train, None).columns)
Xtr_r, ytr_r, wtr_r = expand_r(surv_train, cols_r)
Xva_r, yva_r, wva_r = expand_r(surv_valid, cols_r)
gbm_r = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
    min_child_samples=200, random_state=SEED, verbose=-1)
gbm_r.fit(Xtr_r, ytr_r, sample_weight=wtr_r, eval_set=[(Xva_r, yva_r)],
          eval_sample_weight=[wva_r], callbacks=[lgb.early_stopping(50, verbose=False)])

def survpd_r(df):
    Xl = build_loan_matrix_r(df, cols_r).astype(np.float32).values; nn = len(df); sv = np.ones(nn)
    for t in range(1, H+1):
        sv *= (1 - gbm_r.predict_proba(np.column_stack([Xl, np.full(nn, t, np.float32)]))[:, 1])
    return 1 - sv

pd_r_test = survpd_r(surv_test); pd_r_valid = survpd_r(surv_valid)
cal_r = IsotonicRegression(out_of_bounds='clip').fit(pd_r_valid, y36_valid)
pd_r_cal = cal_r.transform(pd_r_test)
print(f'\nAUC@36 тест 2015: полная={roc_auc_score(y36_test, pd36_gbm):.4f} | '
      f'без rate/grade/sub_grade={roc_auc_score(y36_test, pd_r_test):.4f} | '
      f'Δ={roc_auc_score(y36_test, pd_r_test)-roc_auc_score(y36_test, pd36_gbm):+.4f}')

# H2 на дозревшей денежной выборке с редуцированной PD (ставка остаётся в функции прибыли)
pdc_r = pd_r_cal[money['mask']]
g, b = pv_comp(L, r, n_term, lgd=0.50, disc=0.03)
profit_r = -((1-pdc_r)*g + pdc_r*b)/L
rng = np.random.default_rng(SEED)
r_orders = {'profit/$': np.argsort(profit_r), 'risk(PD)': np.argsort(pdc_r), 'random': rng.permutation(len(L))}
def r_port(o, f): return o[np.cumsum(L[o]) <= f*tot]
print(f"\nH2 с PD БЕЗ внутр. скоринга (disc=3%), n={len(L):,}. Поток, млн $")
print(f"{'бюджет':>7} | " + ' | '.join(f'{k:>10}' for k in r_orders))
for f in [0.10, 0.30, 0.50]:
    print(f'{int(f*100):>6}% | ' + ' | '.join(f'{net[r_port(o, f)].sum()/1e6:>10.1f}' for o in r_orders.values()))
ds = []
for _ in range(2000):
    ix = rng.integers(0, len(L), len(L)); Lb, nb_ = L[ix], net[ix]; t = Lb.sum()
    o1 = np.argsort(profit_r[ix]); o2 = np.argsort(pdc_r[ix])
    ds.append((nb_[o1[np.cumsum(Lb[o1]) <= 0.30*t]].sum() - nb_[o2[np.cumsum(Lb[o2]) <= 0.30*t]].sum())/1e6)
lo, hi = np.percentile(ds, [2.5, 97.5])
print(f'\nВыигрыш profit над risk(PD) @30%: [{lo:+.1f}; {hi:+.1f}]  {"значимо" if lo>0 else "н/з"}')

Исключены из признаков: ['grade', 'int_rate', 'sub_grade']

AUC@36 тест 2015: полная=0.6900 | без rate/grade/sub_grade=0.6679 | Δ=-0.0221

H2 с PD БЕЗ внутр. скоринга (disc=3%), n=283,026. Поток, млн $
 бюджет |   profit/$ |   risk(PD) |     random
    10% |       29.2 |       27.2 |       26.8
    30% |       91.1 |       87.0 |       81.3
    50% |      149.7 |      147.8 |      138.4

Выигрыш profit над risk(PD) @30%: [+1.9; +6.3]  значимо


### 7.3 Перебор гиперпараметров GBM-hazard (§2.8)
Диапазон val-AUC@36 по сетке (num_leaves × learning_rate × min_child_samples). 

In [39]:
Xtr_g, ytr_g, wtr_g = expand_person_period(surv_train, hazard_cols)
Xva_g, yva_g, wva_g = expand_person_period(surv_valid, hazard_cols)
y36_va_g = y_default_within_H(surv_valid)
base_hp = dict(num_leaves=63, learning_rate=0.05, min_child_samples=200)
full = [dict(num_leaves=nl, learning_rate=lr_, min_child_samples=mcs)
        for nl in [31, 63, 127] for lr_ in [0.03, 0.05, 0.1] for mcs in [100, 200]]
grid = [base_hp] + [hp for hp in full if hp != base_hp]
print(f'Конфигов: {len(grid)} (val=2014, ранняя остановка).')
aucs = []
for hp in grid:
    mdl = lgb.LGBMClassifier(n_estimators=600, random_state=SEED, verbose=-1, **hp)
    mdl.fit(Xtr_g, ytr_g, sample_weight=wtr_g, eval_set=[(Xva_g, yva_g)],
            eval_sample_weight=[wva_g], callbacks=[lgb.early_stopping(50, verbose=False)])
    a = roc_auc_score(y36_va_g, survival_pd_cols(lambda X: mdl.predict_proba(X)[:, 1], surv_valid, hazard_cols))
    aucs.append(a); print(f"  {hp}  val-AUC@36={a:.4f}{' <- базовая' if hp==base_hp else ''}")
aucs = np.array(aucs)
print(f'\nДиапазон val-AUC@36: [{aucs.min():.4f}; {aucs.max():.4f}]  размах={aucs.max()-aucs.min():.4f}')

Конфигов: 18 (val=2014, ранняя остановка).
  {'num_leaves': 63, 'learning_rate': 0.05, 'min_child_samples': 200}  val-AUC@36=0.6770 <- базовая
  {'num_leaves': 31, 'learning_rate': 0.03, 'min_child_samples': 100}  val-AUC@36=0.6773
  {'num_leaves': 31, 'learning_rate': 0.03, 'min_child_samples': 200}  val-AUC@36=0.6775
  {'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 100}  val-AUC@36=0.6771
  {'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 200}  val-AUC@36=0.6771
  {'num_leaves': 31, 'learning_rate': 0.1, 'min_child_samples': 100}  val-AUC@36=0.6763
  {'num_leaves': 31, 'learning_rate': 0.1, 'min_child_samples': 200}  val-AUC@36=0.6765
  {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 100}  val-AUC@36=0.6766
  {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 200}  val-AUC@36=0.6773
  {'num_leaves': 63, 'learning_rate': 0.05, 'min_child_samples': 100}  val-AUC@36=0.6767
  {'num_leaves': 63, 'learning_rate': 0.1, 'min_child_samp

### 7.4 H2 на 60-месячных: чистый сплит train 2012 (80 %) / cal 2012 (20 %) / test 2013 (§3.4)
Проверка обобщаемости H2 на длинные кредиты (в основных тестах денежная выборка — только 36-мес).

In [40]:
d12 = surv[surv.issue_year == 2012].reset_index(drop=True)
rng = np.random.default_rng(SEED); perm = rng.permutation(len(d12)); cut = int(0.8*len(d12))
tr60 = d12.iloc[perm[:cut]]; ca60 = d12.iloc[perm[cut:]]; te60 = surv[surv.issue_year == 2013]
cols60 = list(build_loan_matrix(tr60, None).columns)
Xtr60, ytr60, wtr60 = expand_person_period(tr60, cols60)
mdl60 = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=200, random_state=SEED, verbose=-1).fit(Xtr60, ytr60, sample_weight=wtr60)
pred60 = lambda X: mdl60.predict_proba(X)[:, 1]
cal60 = IsotonicRegression(out_of_bounds='clip').fit(survival_pd_cols(pred60, ca60, cols60), y_default_within_H(ca60))
pd_te60 = cal60.transform(survival_pd_cols(pred60, te60, cols60))
print(f'Контроль: AUC@36 тест 2013 (все сроки) = {roc_auc_score(y_default_within_H(te60), pd_te60):.4f}')

mob = te60['mob'].values; term = te60['term_months'].values
matok = (mob >= term) & te60['loan_status'].isin(RES_OK).values

def h2_on(sub):
    Lp = te60['loan_amnt'].values[sub].astype(float); rp = te60['int_rate'].values[sub]/100
    npp = te60['term_months'].values[sub].astype(float)
    netp = te60['total_pymnt'].values[sub] + te60['recoveries'].fillna(0).values[sub] - Lp
    pdc = pd_te60[sub]; yv = ((te60['event'].values[sub] == 1) & (te60['duration'].values[sub] <= H)).astype(int)
    g, b = pv_comp(Lp, rp, npp, lgd=0.50, disc=0.03); profit = -((1-pdc)*g + pdc*b)/Lp; tot_p = Lp.sum()
    out = {}
    for f in [0.10, 0.30]:
        op = np.argsort(profit); orr = np.argsort(pdc)
        fp = netp[op[np.cumsum(Lp[op]) <= f*tot_p]].sum()/1e6
        fr = netp[orr[np.cumsum(Lp[orr]) <= f*tot_p]].sum()/1e6
        ds = []
        for _ in range(400):
            ix = rng.integers(0, len(Lp), len(Lp)); Lb, nb_ = Lp[ix], netp[ix]; t = Lb.sum()
            o1 = np.argsort(profit[ix]); o2 = np.argsort(pdc[ix])
            ds.append((nb_[o1[np.cumsum(Lb[o1]) <= f*t]].sum() - nb_[o2[np.cumsum(Lb[o2]) <= f*t]].sum())/1e6)
        lo, hi = np.percentile(ds, [2.5, 97.5]); out[f] = (fp, fr, fp-fr, lo, hi, yv.mean()*100, len(Lp))
    return out

groups = {'все сроки (2013)': matok, '36-мес (2013)': matok & (term == 36), '60-мес (2013)': matok & (term == 60)}
print(f"\n{'подвыборка':>18} | {'n':>7} | {'дефолт@36':>9} | бюджет | {'profit':>7} | {'risk':>7} | {'Δ,млн':>7} | {'95% ДИ':>16} | знач?")
for name, sub in groups.items():
    res = h2_on(sub)
    for f in [0.10, 0.30]:
        fp, fr, d, lo, hi, defr, nn = res[f]
        print(f"{name:>18} | {nn:>7,} | {defr:>8.1f}% |  {int(f*100):>3}% | {fp:>+7.1f} | {fr:>+7.1f} | {d:>+7.1f} | [{lo:+6.1f};{hi:+6.1f}] | {'да' if lo>0 else 'НЕТ'}")

Контроль: AUC@36 тест 2013 (все сроки) = 0.6477

        подвыборка |       n | дефолт@36 | бюджет |  profit |    risk |   Δ,млн |           95% ДИ | знач?
  все сроки (2013) | 134,804 |     14.0% |   10% |   +49.2 |   +22.3 |   +26.9 | [ +25.0; +28.6] | да
  все сроки (2013) | 134,804 |     14.0% |   30% |  +136.2 |   +76.6 |   +59.6 | [ +57.2; +62.1] | да
     36-мес (2013) | 100,422 |     12.3% |   10% |   +20.0 |   +12.2 |    +7.8 | [  +7.1;  +8.6] | да
     36-мес (2013) | 100,422 |     12.3% |   30% |   +55.0 |   +40.9 |   +14.1 | [ +12.9; +15.2] | да
     60-мес (2013) |  34,382 |     18.8% |   10% |   +19.2 |   +14.8 |    +4.5 | [  +3.1;  +5.6] | да
     60-мес (2013) |  34,382 |     18.8% |   30% |   +52.7 |   +46.9 |    +5.8 | [  +3.9;  +7.6] | да
